In [ ]:
import os
import glob
import numpy as np
import pymzml
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

# =========================
# 输入 / 输出
# =========================
PROJECT_ROOT = Path("../..").resolve()

#MZML_DIR = PROJECT_ROOT / "data" / "raw" / "unreadable_mzML"
MZML_DIR = PROJECT_ROOT / "data" / "raw" / "mzML"

OUT_PLOT_DIR = PROJECT_ROOT / "results" / "runs" / "runs_05_01_5" / "mzML_plot_45"
OUT_NPZ_DIR  = PROJECT_ROOT / "results" / "runs" / "runs_05_01_5" / "mzML_npz_45"

os.makedirs(OUT_PLOT_DIR, exist_ok=True)
os.makedirs(OUT_NPZ_DIR, exist_ok=True)

# =========================
# 参数
# =========================
CUT_RT = 60.0
T_END_REFERENCE = 1738.0     # 只作为参考，不再强制补到这里
INTERP_STEP = 0.2
MIN_POINTS_AFTER_CUT = 10

ENABLE_LIGHT_NORM = False
LIGHT_NORM_METHOD = "max"

# 只抢救这两个接近完整的文件
RESCUE_FILES = {
    "UHM125-22430_MS1.mzML",
    "UHM45-22434_MS1.mzML",
}

# 太短的 partial 文件不要
MIN_REQUIRED_MAX_RT_FOR_RESCUE = 1600.0

# =========================
# 找文件
# =========================
files = sorted(glob.glob(os.path.join(str(MZML_DIR), "*.mzML")))

print("Project root:", PROJECT_ROOT)
print("mzML dir:", MZML_DIR)
print("Total mzML files:", len(files))


# =========================
# 工具函数
# =========================
def get_rt_seconds_from_spec(spec):
    try:
        rt = spec.scan_time
    except Exception:
        return None

    if rt is None:
        return None

    unit = "second"

    if isinstance(rt, (tuple, list)):
        if len(rt) == 0:
            return None
        value = rt[0]
        if len(rt) > 1:
            unit = str(rt[1])
    else:
        value = rt

    try:
        value = float(value)
    except Exception:
        return None

    unit_lower = unit.lower()
    if unit_lower.startswith("min"):
        value *= 60.0

    return value


def dedup_strict_increasing(rt, x):
    rt = np.asarray(rt, dtype=np.float32)
    x = np.asarray(x, dtype=np.float32)

    order = np.argsort(rt)
    rt = rt[order]
    x = x[order]

    rt_unique, unique_idx = np.unique(rt, return_index=True)
    x_unique = x[unique_idx]

    return rt_unique.astype(np.float32), x_unique.astype(np.float32)


def light_normalize(x, method="max"):
    x = np.asarray(x, dtype=np.float32)

    if method == "max":
        denom = float(np.max(x))
    elif method == "p99":
        denom = float(np.percentile(x, 99))
    else:
        raise ValueError(f"Unknown LIGHT_NORM_METHOD: {method}")

    if (not np.isfinite(denom)) or (denom <= 0):
        return x

    return (x / (denom + 1e-12)).astype(np.float32)


def save_raw_cut_plot(base, rts_raw, sig_raw, rts_cut, sig_cut, cut_rt, out_plot_dir, partial_read=False):
    plt.figure(figsize=(12, 4))
    plt.plot(rts_raw, sig_raw, label="raw", alpha=0.35, linewidth=1.2)
    plt.plot(rts_cut, sig_cut, label="cut", linewidth=1.5)
    plt.axvline(cut_rt, linestyle="--", linewidth=1.2, label="cut_rt")

    title_suffix = " | PARTIAL RESCUED" if partial_read else ""
    plt.xlabel("Retention time")
    plt.ylabel("TIC")
    plt.title(f"{base} | raw={len(sig_raw)} cut={len(sig_cut)}{title_suffix}")
    plt.legend(loc="upper right")
    plt.tight_layout()

    out_png = os.path.join(out_plot_dir, base.replace(".mzML", "_raw_cut.png"))
    plt.savefig(out_png, dpi=150)
    plt.close()


def save_grid_plot(base, rt_grid, signal_grid, out_plot_dir, partial_read=False):
    plt.figure(figsize=(8, 3.5))
    plt.plot(rt_grid, signal_grid, linewidth=1)

    title_suffix = " | PARTIAL RESCUED" if partial_read else ""
    plt.xlabel("Retention time")
    plt.ylabel("Signal")
    plt.title(f"{base}{title_suffix} | grid={len(signal_grid)}")
    plt.tight_layout()

    out_png = os.path.join(out_plot_dir, base.replace(".mzML", "_grid.png"))
    plt.savefig(out_png, dpi=150)
    plt.close()


# =========================
# 主循环
# =========================
n_ok = 0
n_skip = 0
n_rescued = 0

for f in tqdm(files):
    base = os.path.basename(f)

    rts = []
    sig = []
    parse_error = None
    partial_read = False

    try:
        run = pymzml.run.Reader(f)

        try:
            for spec in run:
                try:
                    ms_level = getattr(spec, "ms_level", None)
                except Exception:
                    ms_level = None

                if ms_level != 1:
                    continue

                rt_sec = get_rt_seconds_from_spec(spec)
                if rt_sec is None:
                    continue

                try:
                    intensity_sum = float(np.sum(spec.i))
                except Exception:
                    continue

                if not np.isfinite(rt_sec) or not np.isfinite(intensity_sum):
                    continue

                rts.append(rt_sec)
                sig.append(intensity_sum)

        except Exception as e:
            parse_error = e
            partial_read = True

            print(f"\n[PARTIAL READ] {base}")
            print(f"  error: {repr(e)}")
            print(f"  n_ms1 before error: {len(rts)}")
            print(f"  last_rt before error: {rts[-1] if len(rts) > 0 else None}")

        if len(rts) == 0:
            print("No usable MS1 scans:", base)
            n_skip += 1
            continue

        max_rt_observed = float(np.max(rts))

        # partial 文件筛选
        if partial_read:
            if base not in RESCUE_FILES:
                print(f"[SKIP PARTIAL] {base} not selected for rescue. max_rt={max_rt_observed}")
                n_skip += 1
                continue

            if max_rt_observed < MIN_REQUIRED_MAX_RT_FOR_RESCUE:
                print(f"[SKIP PARTIAL] {base} too truncated. max_rt={max_rt_observed}")
                n_skip += 1
                continue

            print(f"[RESCUE PARTIAL] {base} | max_rt={max_rt_observed}")
            n_rescued += 1

        # =========================
        # 转 numpy + 排序
        # =========================
        rts = np.asarray(rts, dtype=np.float32)
        sig = np.asarray(sig, dtype=np.float32)

        if rts.shape[0] != sig.shape[0]:
            print("Length mismatch:", base, rts.shape, sig.shape)
            n_skip += 1
            continue

        order = np.argsort(rts)
        rts = rts[order]
        sig = sig[order]

        rt_raw = rts.copy()
        tic_raw = sig.copy()

        # =========================
        # cut: 只保留 CUT_RT 之后，且不超过参考上限
        # 但是不会补到 T_END_REFERENCE
        # =========================
        mask = (rts >= CUT_RT) & (rts <= T_END_REFERENCE)

        if mask.sum() < MIN_POINTS_AFTER_CUT:
            print("Too few points after cut:", base)
            n_skip += 1
            continue

        rt_cut = rts[mask]
        tic_cut = sig[mask]

        rt_cut, tic_cut = dedup_strict_increasing(rt_cut, tic_cut)

        if len(rt_cut) < 2:
            print("Too few unique RT points:", base)
            n_skip += 1
            continue

        # =========================
        # 关键修改：
        # 每个文件只插值到自己的真实 max_rt_cut
        # 不再补到统一 T_END_REFERENCE
        # =========================
        file_rt_start = float(CUT_RT)
        file_rt_end = float(np.max(rt_cut))

        rt_grid_file = np.arange(
            file_rt_start,
            file_rt_end + INTERP_STEP,
            INTERP_STEP,
            dtype=np.float32
        )

        # 防止 arange 最后超过真实观测末尾太多
        rt_grid_file = rt_grid_file[rt_grid_file <= file_rt_end]

        if len(rt_grid_file) < 2:
            print("Too few grid points:", base)
            n_skip += 1
            continue

        if ENABLE_LIGHT_NORM:
            signal_for_interp = light_normalize(tic_cut, method=LIGHT_NORM_METHOD)
        else:
            signal_for_interp = tic_cut.astype(np.float32)

        # 只在真实观测范围内插值，不设置 right=0
        signal_grid = np.interp(
            rt_grid_file,
            rt_cut,
            signal_for_interp
        ).astype(np.float32)

        # =========================
        # 画图
        # =========================
        save_raw_cut_plot(
            base,
            rt_raw,
            tic_raw,
            rt_cut,
            tic_cut,
            CUT_RT,
            OUT_PLOT_DIR,
            partial_read=partial_read
        )

        save_grid_plot(
            base,
            rt_grid_file,
            signal_grid,
            OUT_PLOT_DIR,
            partial_read=partial_read
        )

        # =========================
        # 保存 meta
        # =========================
        meta = {
            "filename": base,
            "cut_rt": float(CUT_RT),
            "t_end_reference": float(T_END_REFERENCE),
            "actual_rt_end": float(file_rt_end),
            "interp_step": float(INTERP_STEP),

            "enable_light_norm": bool(ENABLE_LIGHT_NORM),
            "light_norm_method": LIGHT_NORM_METHOD if ENABLE_LIGHT_NORM else "none",

            "n_points_raw": int(len(rt_raw)),
            "n_points_cut": int(len(rt_cut)),
            "n_points_grid": int(len(rt_grid_file)),

            "partial_read": bool(partial_read),
            "rescued_partial_file": bool(partial_read and base in RESCUE_FILES),
            "parse_error": repr(parse_error) if parse_error is not None else "",

            "max_rt_observed_raw": float(max_rt_observed),
            "max_rt_cut": float(file_rt_end),

            "not_padded_to_reference_end": True,
            "missing_tail_seconds_vs_reference": float(max(0.0, T_END_REFERENCE - file_rt_end)),
        }

        # =========================
        # 保存 npz
        # =========================
        npz_name = base.replace(".mzML", ".npz")
        npz_path = os.path.join(OUT_NPZ_DIR, npz_name)

        np.savez_compressed(
            npz_path,

            rt_raw=rt_raw.astype(np.float32),
            tic_raw=tic_raw.astype(np.float32),

            rt_cut=rt_cut.astype(np.float32),
            tic_cut=tic_cut.astype(np.float32),

            signal_for_interp=signal_for_interp.astype(np.float32),

            # 每个文件自己的 grid
            rt_grid=rt_grid_file.astype(np.float32),
            signal_grid=signal_grid.astype(np.float32),

            # 兼容旧 pipeline
            rt=rt_grid_file.astype(np.float32),
            signal=signal_grid.astype(np.float32),

            meta=meta
        )

        n_ok += 1
        print(f"[SAVED] {base} | grid_len={len(rt_grid_file)} | rt_end={file_rt_end:.3f}")

    except Exception as e:
        print("Error:", base, repr(e))
        n_skip += 1


print("\nDone.")
print("NPZ saved to:", OUT_NPZ_DIR)
print("Plots saved to:", OUT_PLOT_DIR)
print(f"Success: {n_ok} | Rescued partial: {n_rescued} | Skipped/Error: {n_skip}")